In [434]:
import pandas as pd
import bs4 as bs
import re
from group import RateGroup, RateAgreement, RateSteps
import json
from dateutil import parser
from datetime import datetime

In [5]:
classDf = pd.read_pickle('classDf.pkl')
idContentDf = pd.read_pickle('idContentDf.pkl')

In [661]:
def getRateTables(tbsId, appendixText):
    ac = idContentDf.loc[idContentDf['ids'] == tbsId, 'htmlContent'].values[0].decode('utf-8', 'ignore')
 
    ac = ac.replace('“', '')
    ac = ac.replace('”', '|')
    ac = ac.replace(u'\xa0', ' ') #non breaking space
    ac = ac.replace('&#160;&#8211; ', '-') #non breaking space
    ac = ac.replace('&#8211; ', '-') #non breaking space

    #print(ac)

    soup = bs.BeautifulSoup(ac,'lxml')

    tables = soup.select('h2:contains("%s") ~ table' %(appendixText))

    tables = tables + soup.select('h2:contains("%s") ~ section table' %(appendixText))

    return tables

In [677]:
def parseHTMLToJSON(tbsId, rateTables, sep):
    rates = []
    for rateCat in rateTables:
        #print(rateCat)
        steps = rateCat.select('thead th')

        if len(steps) == 0:
            #print('no steps found')
            steps = rateCat.select('table:not(tfoot) > tr')[0].select('th')

        #print(steps)

        agreementTable = rateCat.select('tbody tr')
        if len(agreementTable) == 0:
            #print('no tbody found')
            agreementTable = rateCat.select('table:not(tfoot) > tr')
            agreementTable.pop(0)

        #print(agreementTable)
        #print(len(agreementTable))
        rateAgreements = getAgreements(agreementTable, steps) #might be able to detect if tbody is present here and remove column 1 removing the need for step + 1

        withSepGrpLvl = re.split(' annual', rateCat.find('caption').getText(), flags=re.IGNORECASE)
        grpLvl = withSepGrpLvl[0].rstrip(sep).strip()
        
        #print(grpLvl)
        #print(grpLvl.split('-'))

        if len(grpLvl.split('-')) == 1:
            grp = grpLvl.split(sep)[0].strip()
            lvl = grpLvl.split(sep)[1].strip()
        else:
            grp = grpLvl.split('-')[0].strip()
            lvl = grpLvl.split('-')[1]
            if (lvl == '00'):
                lvl = '0'
            else:
                lvl = grpLvl.split('-')[1].lstrip('0').lstrip(sep).split('/')[0].strip()

        #print(grp)
        #print(lvl)
        rateGrp = RateGroup(
            grp,
            lvl,       
            rateAgreements
        )
        rates.append(rateGrp)

    results = [rateGrp.to_dict() for rateGrp in rates]
    #print(results)
    with open('data/' + tbsId + '.json', 'w') as json_file:
        json.dump(results, json_file)
    return results

In [670]:
def getAgreements(agreementTable, steps):
    rateAgreements = []
    for agreement in agreementTable:
        #print(agreement)
        rateStepsList = []
        amounts = agreement.findAll('td') #all the cells inside each row - since the row names are th we can get amts only with td
        for idx,amount in enumerate(amounts):
            if amount.find(' to ') == -1:
                newAmts = [amountgetText().replace(',', '')]
            else:
                newAmts = amount.getText().replace(',', '').split(' to ')

            step = re.search('Step.[0-9]*', steps[idx+1].getText())
            if step is None:
                agStep = steps[idx+1].getText().replace('Range', '1') #if its just one level and says range instead of a level
                agSteg = int(agStep)
            else:
                agStep = int(re.search('[0-9]',step[0])[0])

            agAmts = []
            for amt in newAmts:
                if amt.isdigit():
                    agAmts.append(int(amt))
            if len(agAmts) > 0:
                rateStep = RateSteps(agStep, agAmts)
                rateStepsList.append(rateStep)

        #Getting the date
        rateAgreementDate  = agreement.find('time')
        #print(rateAgreementDate)
        #Date not always available as an attr
        if rateAgreementDate is None:
            dateStr = agreement.find('th').contents[0].split(')')[1].lstrip(' Wage Adjustment - ')
            #print(dateStr)
            rateAgreementDate = parser.parse(dateStr).date()
            rateAgreementDate = rateAgreementDate.strftime('%Y-%m-%d')
        else:
            rateAgreementDate = rateAgreementDate.get('datetime')
        
        rateAgreement = RateAgreement(rateAgreementDate, rateStepsList)
        rateAgreements.append(rateAgreement)

    return rateAgreements

In [8]:
def getJSON(tbsId, appendixStr, sep):
    rateTables = getRateTables(tbsId, appendixStr)
    return parseHTMLToJSON(tbsId, rateTables, sep)

In [681]:
page_19 = getJSON('19', '**Appendix A', ':')

Restructure will be effective within 180 days after the signing of the collective agreement


ParserError: Unknown string format: Restructure will be effective within 180 days after the signing of the collective agreement

In [664]:
page_1 = getJSON('1', '**Appendix A', ':')
page_2 = getJSON('2', '**Appendix A', ':')
page_3 = getJSON('3', '**Appendix A', ':')
page_4 = getJSON('4', '**Appendix A', '-')
page_5 = getJSON('5', '**Appendix A', ':')
page_7 = getJSON('7', '** Appendix A', ':')
page_9 = getJSON('9', '**Appendix B-1', ':')
page_10 = getJSON('10', '** Appendix A', ':')
page_11 = getJSON('11', '**Appendix A', ' - ')
page_12 = getJSON('12', '**Appendix A', ':')
page_14 = getJSON('14', '**Appendix A', ':') #kinda works
page_15 = getJSON('15', '**Appendix A‑1', '-') #couple of grp-lvls not working
page_16 = getJSON('16', '**Appendix A', ':')
page_16 = getJSON('17', '**Appendix A', ':')
page_18 = getJSON('18', '**Appendix A', ':')
page_26 = getJSON('26', '**Appendix A', ' - ')
page_27 = getJSON('27', '**Appendix A', ':')

<tr>
<th class="active norwap" scope="row">$) <time class="nowrap" datetime="2013-12-22">December 22, 2013</time></th>
<td class="text-right nowrap">53611</td>
<td class="text-right nowrap">55593</td>
<td class="text-right nowrap">57573</td>
<td class="text-right nowrap">59541</td>
<td class="text-right nowrap">61508</td>
<td class="text-right nowrap">63474</td>
<td class="text-right nowrap">65439</td>
<td class="text-right nowrap">69088</td>
</tr>
<time class="nowrap" datetime="2013-12-22">December 22, 2013</time>
<tr>
<th class="active norwap" scope="row">A) <time class="nowrap" datetime="2014-12-22">December 22, 2014</time></th>
<td class="text-right nowrap">54281</td>
<td class="text-right nowrap">56288</td>
<td class="text-right nowrap">58293</td>
<td class="text-right nowrap">60285</td>
<td class="text-right nowrap">62277</td>
<td class="text-right nowrap">64267</td>
<td class="text-right nowrap">66257</td>
<td class="text-right nowrap">69952</td>
</tr>
<time class="nowrap" datet

In [226]:
with open('data/combined/payscales.json', 'w') as json_file:
        json.dump(combined_list, json_file)

In [389]:
#getJSON('14', '**Appendix A', ':') -> yeah this won't work...
#getJSON('8', '**Appendix A', ' - ') -> yeah this ALSO won't work
#getJSON('6', '**Appendix A', ' - ') -> Y) Restructure messing me up
#getJSON('13', '**Appendix A', ' - ') -> region specific